# Patent Image Extraction and Processing Pipeline
## Master Thesis Project: eVTOL/UAM Image Dataset

### Objetivo
Este notebook foi reorganizado em módulos independentes para facilitar a extração futura para múltiplos notebooks (`.ipynb`) e reduzir o acoplamento entre configuração, leitura de dados, inferência e processamento geométrico.

### Estrutura Modular
1. **Módulo 1: Configuração e Utils**
   - Paths, thresholds, constantes centrais
   - Utils de rede/sistema
   - Funções geométricas atómicas de imagem
2. **Módulo 2: Leitura e Estado**
   - Leitura do Excel
   - Extração de hyperlinks
   - Construção do tracker mestre
3. **Módulo 3: Extração**
   - Motor de inferência YOLO
   - Download/renderização de PDFs
   - Crop e processamento de imagem
4. **Módulo 4: Resumo e Exportação**
   - Consolidação de resultados
   - Exportação de JSON/CSV
   - DataFrames limpos para visualização

### Heurísticas Principais
- `confidence_threshold`: confiança principal do YOLO
- `fallback_confidence_threshold`: confiança de recuperação por página
- `target_classes`: classes alvo para extração
- `min_crop_size`: filtro para crops inválidos ou demasiado pequenos

### Nota
O código mantém a funcionalidade original, mas agora cada bloco possui fronteiras claras para ser movido para um notebook próprio sem reescrita substancial.

In [ ]:
# Minimal pipeline path configuration (override PROJECT_ROOT if needed)
from pathlib import Path
import platform
import os

OS_TYPE = platform.system()
if OS_TYPE == "Linux":
    PROJECT_ROOT = Path('/home/vasco/Tese_Vasco_Lnx/Drive_files_to_syncronize/3 - Images Processing & DataSets/2-Images_DataSets/Test for last Query DataSet Extraction')
elif OS_TYPE == "Windows":
    PROJECT_ROOT = Path(r'C:\path\to\imagens_aprovadas_main')
else:
    PROJECT_ROOT = Path.home() / 'imagens_aprovadas_main'

# Keys below are referenced by later cells; keep names stable.
PATHS = {
    "output_images": PROJECT_ROOT / "Raw_extracted_images",
    "output_approved": PROJECT_ROOT / "Approved_images",
}

for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

print('CONFIG: PROJECT_ROOT =', PROJECT_ROOT)
for k, v in PATHS.items():
    print(f'  - {k}: {v}')


In [ ]:
# ==============================================================================
# MÓDULO 1: CONFIGURAÇÃO GLOBAL, UTILS DE REDE/SISTEMA E PROCESSAMENTO GEOMÉTRICO
# ==============================================================================
# RESPONSABILIDADES:
#   - Imports: cv2 (OpenCV), PyMuPDF (renderizar PDF), requests, PIL, YOLOv10
#   - Configurações: paths, thresholds YOLO, constantes PDF
#   - Utils: download HTTP com retry, procura de ficheiros, normalização IDs
#   - Geométrico: detecção cor bordas, resize aspect-ratio, padding canvas
# FERRAMENTAS PRINCIPAIS:
#   - requests/urllib3: Downloads com retry automático
#   - pymupdf (fitz): Renderizar páginas PDF -> imagem OpenCV
#   - cv2 (OpenCV): Converter color-space, manipulação imagem
#   - PIL/Pillow: Resize com LANCZOS, criar canvas, trabalho RGB
#   - doclayout_yolo: Inferência de detecção de objetos
# ==============================================================================

import json
import os
import warnings
from datetime import datetime
from pathlib import Path

# === FERRAMENTAS DE IMAGEM ===
import cv2  # OpenCV: conversão color-space, slicing, operações numpy
import numpy as np  # Cálculos: mediana bordas, clipping valores
import pandas as pd  # Estruturas dados: DataFrames
from PIL import Image  # PIL/Pillow: resize com interpolação, canvas, conversão RGB

# === FERRAMENTAS DE PDF ===
import pymupdf as fitz  # PyMuPDF: renderizar PDF -> pixel array

# === FERRAMENTAS DE REDE ===
import requests  # HTTP requests: download PDFs
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry  # Retry automático para timeouts/erros

# === FERRAMENTAS DE ML ===
import torch  # PyTorch: detectar GPU disponível
from doclayout_yolo import YOLOv10  # YOLO v10: detector de layout de documentos

# === FERRAMENTAS DE I/O ===
from openpyxl import load_workbook  # Ler Excel com metadados (hyperlinks)

warnings.filterwarnings("ignore")

print("=" * 80)
print("✅ DEPENDÊNCIAS IMPORTADAS COM SUCESSO")
print("=" * 80)



# === CONFIGURAÇÃO: THRESHOLDS DE INFERÊNCIA YOLO ===
# - confidence_threshold: confiança mínima para aceitar detecção (mais alto = mais rigoroso)
# - fallback_confidence_threshold: confiança de recuperação se página não tem detecções
# - target_classes: apenas estas classes são extraídas como crops
# - min_crop_size: rejeita crops menores que este tamanho (pixels)
MODEL_CONFIG = {
    "name": "doclayout_yolo_docstructbench_imgsz1024.pt",
    "confidence_threshold": 0.35,  # 35% - balanço entre ruído e recall
    "fallback_confidence_threshold": 0.18,  # 18% - recuperação em páginas difíceis
    "target_classes": ["figure", "picture", "image"],  # Classes alvo
    "min_crop_size": 24,  # Rejeita crops < 24px (muito pequeno)
}

# === CONFIGURAÇÃO: RENDERIZAÇÃO PDF E REQUISIÇÕES HTTP ===
# - zoom_extraction: 2.0x amplifica resolução (melhor OCR, mais detalhes)
# - batch_size: número de páginas processadas de uma vez
# - timeout: segundos máximos por download
# - user_agent: simula browser para evitar bloqueios
PDF_CONFIG = {
    "zoom_extraction": 2.0,  # 2x: renderiza em 2000x2600px (vs 1000x1300px)
    "batch_size": 4,  # Processa 4 páginas por batch (otimização VRAM)
    "timeout": 60,  # 60 segundos antes de timeout
    "user_agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
}

# === CONFIGURAÇÃO: METADADOS EXCEL ===
EXCEL_CONFIG = {
    "sheet_index": 0,
    "id_column": "Record Number",  # Identificador único da patente
    "link_column": "PDF Link",  # URL do PDF
}

for path in PATHS.values():
    path.mkdir(exist_ok=True, parents=True)

print(f"Working directory: {os.getcwd()}")
print(f"GPU available: {'YES ✅' if torch.cuda.is_available() else 'NO (CPU)'}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print("=" * 80 + "\n")


# ==============================================================================
# UTILS: SISTEMA E FICHEIROS
# ==============================================================================

def find_file(filename, search_paths=None):
    """Localiza ficheiro em paths estruturados (ferramentas: Path, os.walk)."""
    if search_paths is None:
        tese_root = SCRIPT_DIR
        while tese_root.name and "Tese Vasco" not in tese_root.name:
            tese_root = tese_root.parent
        if "Tese Vasco" not in tese_root.name:
            tese_root = SCRIPT_DIR.parent.parent.parent

        search_paths = [
            Path.cwd(),
            SCRIPT_DIR,
            SCRIPT_DIR.parent,
            SCRIPT_DIR.parent / "DocLayout-YOLO",
            tese_root,
            tese_root / "3- DataSet Validation" / "DataSets",
            tese_root / "1- Literatura",
        ]

    if Path(filename).is_file():
        return str(Path(filename).resolve())

    for search_dir in search_paths:
        search_dir = Path(search_dir)
        if not search_dir.exists():
            continue
        candidate = search_dir / filename
        if candidate.is_file():
            return str(candidate.resolve())

    for search_dir in search_paths:
        search_dir = Path(search_dir)
        if not search_dir.exists():
            continue
        for root, _, files in os.walk(search_dir):
            if filename in files:
                return str((Path(root) / filename).resolve())

    raise FileNotFoundError(f"Could not locate '{filename}'")


def normalize_patent_id(patent_id):
    """Normaliza ID de patente (upper, strip) para matching estável."""
    return str(patent_id).strip().upper()


# ==============================================================================
# UTILS: REDE (DOWNLOAD COM RETRY)
# ==============================================================================

def create_http_session_with_retry():
    """Cria sessão HTTP com retry automático (ferramentas: requests, urllib3.Retry).
    
    Retenta em caso de:
      - 429 (Rate Limited)
      - 500-504 (Server Errors)
    
    Backoff exponencial: 2x delay entre tentativas.
    """
    session = requests.Session()
    retry_strategy = Retry(
        total=5,  # Máximo 5 retentativas
        backoff_factor=2,  # 1s, 2s, 4s, 8s, 16s
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "HEAD"],
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


def download_pdf_bytes(url, session=None, timeout=30):
    """Descarrega PDF bytes com validação (ferramentas: requests, magic bytes).
    
    Validações:
      - Tamanho mínimo 1KB (rejeita ficheiros truncados)
      - Magic bytes: começa com b'%PDF' ou Content-Type contém 'pdf'
    
    Retorna: bytes ou None se falha.
    """
    if session is None:
        session = create_http_session_with_retry()

    try:
        headers = {"User-Agent": PDF_CONFIG["user_agent"]}
        response = session.get(url, headers=headers, timeout=timeout, stream=True)
        response.raise_for_status()

        content = bytearray()
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                content.extend(chunk)

        pdf_bytes = bytes(content)
        
        # Validação 1: tamanho mínimo
        if len(pdf_bytes) < 1024:
            print("      ⚠️ Downloaded file is too small")
            return None

        # Validação 2: magic bytes ou Content-Type
        if not pdf_bytes.startswith(b"%PDF"):
            content_type = response.headers.get("Content-Type", "")
            if "pdf" not in content_type.lower():
                print(f"      ⚠️ Non-PDF response received (Content-Type: {content_type})")
                return None

        return pdf_bytes
    except Exception as e:
        print(f"      ❌ Download failed: {str(e)[:120]}")
        return None


# ==============================================================================
# UTILS: EXCEL (EXTRAÇÃO DE HYPERLINKS)
# ==============================================================================

def extract_hyperlinks_from_excel(excel_path, column_name, sheet_index=0):
    """Extrai URLs reais de coluna Excel (ferramentas: openpyxl).
    
    Prioridade:
      1. cell.hyperlink.target (URL verdadeira armazenada)
      2. cell.value se começa com http/https/ftp (texto visível)
      3. None se célula vazia ou sem URL
    
    Retorna: lista de URLs (com None para células inválidas).
    """
    try:
        workbook = load_workbook(excel_path)
        worksheet = workbook.worksheets[sheet_index]

        col_idx = None
        for col_num, cell in enumerate(worksheet[1], 1):
            if cell.value == column_name:
                col_idx = col_num
                break

        if col_idx is None:
            print(f"Column '{column_name}' not found in Excel")
            return []

        urls = []
        for row_num in range(2, worksheet.max_row + 1):
            cell = worksheet.cell(row=row_num, column=col_idx)
            if cell.hyperlink and cell.hyperlink.target:
                urls.append(cell.hyperlink.target)
            elif cell.value and isinstance(cell.value, str):
                if cell.value.startswith(("http", "https", "ftp")):
                    urls.append(cell.value.strip())
                else:
                    urls.append(None)
            else:
                urls.append(None)

        return urls
    except Exception as e:
        print(f"Error extracting hyperlinks: {e}")
        return []


# ==============================================================================
# UTILS: CARREGAMENTO MODELO YOLO
# ==============================================================================

def load_model_with_fallback():
    """Carrega modelo YOLO (ferramentas: YOLOv10, find_file).
    
    Fluxo:
      1. Procura modelo configurado em disco (find_file)
      2. Se não encontra, fallback para yolov10n.pt pré-treinado
    
    Retorna: modelo YOLOv10 pronto para inferência.
    """
    model_name = MODEL_CONFIG["name"]
    try:
        model_path = find_file(model_name)
        print(f"Loading YOLO from: {model_path}")
        return YOLOv10(model_path)
    except FileNotFoundError:
        print("Model not found, using default detection...")
        return YOLOv10("yolov10n.pt")


# ==============================================================================
# UTILS: RENDERIZAÇÃO PDF
# ==============================================================================

def render_pdf_page_to_bgr(page, zoom):
    """Renderiza página PDF em imagem OpenCV BGR (ferramentas: PyMuPDF, cv2).
    
    Etapas:
      1. PyMuPDF (fitz): matriz de zoom -> pixmap (PIL) com samples
      2. NumPy: reshape samples em array H×W×3 ou H×W×4
      3. cv2: converter para BGR (OpenCV standard)
    
    Retorna: numpy array HxWx3 em BGR (pronto para YOLO/OpenCV).
    """
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False, annots=False)
    img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)

    if pix.n == 1:
        return cv2.cvtColor(img_array, cv2.COLOR_GRAY2BGR)
    if pix.n == 4:
        return cv2.cvtColor(img_array, cv2.COLOR_BGRA2BGR)
    return cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)


# ==============================================================================
# UTILS: INFERÊNCIA YOLO (COM FALLBACK PÁGINA-A-PÁGINA)
# ==============================================================================

def run_yolo_inference_with_fallback(model, images_cv, confidence, device):
    """Executa YOLO em batch com fallback page-by-page (ferramentas: YOLOv10).
    
    Estratégia:
      1. Tenta batch (eficiente em VRAM)
      2. Se falha (OOM), retenta página individual
    
    Retorna: (lista de resultados YOLO, flag se usou fallback).
    """
    if not images_cv:
        return [], False

    try:
        return model(images_cv, verbose=False, conf=confidence, device=device), False
    except Exception as e:
        print(f"      ⚠️ Batch inference failed ({str(e)[:100]}), trying page-by-page...")
        fallback_results = []
        for image_cv in images_cv:
            try:
                single_result = model([image_cv], verbose=False, conf=confidence, device=device)
                fallback_results.append(single_result[0] if len(single_result) > 0 else None)
            except Exception:
                fallback_results.append(None)
        return fallback_results, True


# ==============================================================================
# UTILS: EXTRAÇÃO BOUNDING BOXES (APENAS CLASSES-ALVO)
# ==============================================================================

def extract_target_boxes(yolo_result, target_classes):
    """Filtra detecções YOLO por classe (ferramentas: YOLOv10 resultado).
    
    Fluxo:
      1. Valida resultado YOLO (não None, tem .boxes)
      2. Itera cada box, extrai class_idx
      3. Compara com target_classes (case-insensitive)
      4. Retorna apenas target boxes com (x1,y1,x2,y2,conf)
    
    Retorna: lista de tuplos (class_name, x1, y1, x2, y2, confidence).
    """
    if yolo_result is None or not hasattr(yolo_result, "boxes") or yolo_result.boxes is None:
        return []

    extracted = []
    target_set = {c.lower() for c in target_classes}

    for box in yolo_result.boxes:
        class_idx = int(box.cls[0])
        class_name = str(yolo_result.names[class_idx]).lower()
        if class_name not in target_set:
            continue

        x1, y1, x2, y2 = map(float, box.xyxy[0])
        confidence = float(box.conf[0]) if hasattr(box, "conf") else None
        extracted.append((class_name, x1, y1, x2, y2, confidence))

    return extracted


# ==============================================================================
# UTILS: SANITIZAÇÃO BOUNDING BOXES (VALIDAÇÃO GEOMÉTRICA)
# ==============================================================================

def sanitize_bbox(x1, y1, x2, y2, img_width, img_height, min_size=16, pad=2):
    """Valida e clampeia bounding box aos limites da imagem (ferramentas: numpy).
    
    Validações:
      1. Padding de 2px nas bordas (contexto adicional)
      2. Clamp aos limites [0, width] e [0, height]
      3. Rejeita se inverted (x2 <= x1) ou muito pequeno (< min_size)
    
    Retorna: (x1, y1, x2, y2) válidos ou None se inválido.
    """
    x1 = max(0, int(np.floor(x1)) - pad)
    y1 = max(0, int(np.floor(y1)) - pad)
    x2 = min(img_width, int(np.ceil(x2)) + pad)
    y2 = min(img_height, int(np.ceil(y2)) + pad)

    if x2 <= x1 or y2 <= y1:
        return None
    if (x2 - x1) < min_size or (y2 - y1) < min_size:
        return None
    return x1, y1, x2, y2


# ==============================================================================
# PROCESSAMENTO GEOMÉTRICO: DETECÇÃO COR DE FUNDO (PADDING)
# ==============================================================================

def calculate_background_color(image, inset_px=8):
    """Deteta cor dominante nas bordas da imagem (ferramentas: PIL, numpy).
    
    Algoritmo:
      1. Amostra 4 bordas (topo, base, esquerda, direita) com inset_px
      2. Concatena pixels das bordas
      3. Calcula mediana RGB
      4. Estima luminância (0.299R + 0.587G + 0.114B)
    
    Regras:
      - Se luminância > 200: branco puro (fundo claro comum em patents)
      - Se luminância < 30: preto puro (fundo muito escuro)
      - Senão: devolve cor média (grey/dark)
    
    Retorna: tuplo (R, G, B) para usar em canvas padding.
    """
    img = image.convert("RGB")
    arr = np.asarray(img)
    h, w = arr.shape[:2]

    if h == 0 or w == 0:
        return (255, 255, 255)

    border = max(1, min(int(inset_px), max(1, min(h, w) // 10)))
    top = arr[:border, :]
    bottom = arr[-border:, :]
    left = arr[:, :border]
    right = arr[:, -border:]

    bands = [band.reshape(-1, 3) for band in [top, bottom, left, right] if band.size > 0]
    if not bands:
        return (255, 255, 255)

    border_pixels = np.concatenate(bands, axis=0)
    if border_pixels.size == 0:
        return (255, 255, 255)

    med_color = np.median(border_pixels, axis=0)
    luminance = 0.299 * med_color[0] + 0.587 * med_color[1] + 0.114 * med_color[2]

    if luminance > 200:
        return (255, 255, 255)
    if luminance < 30:
        return (0, 0, 0)
    return tuple(int(np.clip(c, 0, 255)) for c in med_color)


# ==============================================================================
# PROCESSAMENTO GEOMÉTRICO: RESIZE COM ASPECT-RATIO
# ==============================================================================

def resize_maintaining_aspect(image, size=256):
    """Redimensiona PIL image mantendo aspect-ratio (ferramentas: PIL LANCZOS).
    
    Etapas:
      1. Converte para RGB (caso seja RGBA, grayscale, etc)
      2. Calcula scale: min(target_w/src_w, target_h/src_h)
      3. Aplica resize com interpolação LANCZOS (melhor qualidade)
    
    Resultado: imagem PIL redimensionada mas não-quadrada (ainda precisa padding).
    
    Retorna: PIL Image com altura/largura <= size.
    """
    if not isinstance(image, Image.Image):
        raise TypeError("image must be a PIL.Image.Image instance")

    target = int(size)
    img = image.convert("RGB")
    src_w, src_h = img.size
    scale = min(target / src_w, target / src_h)  # Mantém aspect-ratio

    new_w = max(1, int(round(src_w * scale)))
    new_h = max(1, int(round(src_h * scale)))
    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)


# ==============================================================================
# PROCESSAMENTO GEOMÉTRICO: PADDING (CANVAS QUADRADO)
# ==============================================================================

def apply_canvas_padding(image, size=256, fill_color=(0, 0, 0)):
    """Centra imagem em canvas quadrado com cor de fundo (ferramentas: PIL).
    
    Etapas:
      1. Cria canvas quadrado size×size com fill_color (RGB)
      2. Centra imagem no canvas: offset = (size - img_w) // 2
      3. Cola imagem no canvas (PIL.paste)
    
    Resultado: imagem quadrada size×size com imagem centrada + padding.
    
    Ferramentas:
      - Image.new(): criar canvas RGB
      - canvas.paste(): colar imagem no offset
    
    Retorna: PIL Image quadrada size×size.
    """
    if not isinstance(image, Image.Image):
        raise TypeError("image must be a PIL.Image.Image instance")

    target = int(size)
    canvas = Image.new("RGB", (target, target), fill_color)
    offset_x = (target - image.size[0]) // 2
    offset_y = (target - image.size[1]) // 2
    canvas.paste(image, (offset_x, offset_y))
    return canvas


# ==============================================================================
# PIPELINE COMPLETO: ORQUESTRA TODAS AS ETAPAS GEOMÉTRICAS
# ==============================================================================

def process_image_pipeline(image, size=256):
    """Executa pipeline completo: cor bordas -> resize aspect -> padding canvas.
    
    Etapas (nesta ordem):
      1. calculate_background_color(): amostra bordas, deteta cor RGB
      2. resize_maintaining_aspect(): redimensiona com aspect-ratio
      3. apply_canvas_padding(): centra em canvas quadrado com cor detectada
    
    Fluxo visual:
      [Imagem original] 
        -> [Amostra bordas] 
        -> [Determina fill_color]
        -> [Resize proporcional a 256x?]
        -> [Cria canvas 256x256 com fill_color]
        -> [Cola imagem centrada]
        -> [Resultado: 256x256 com padding]
    
    Retorna: PIL Image quadrada size×size pronta para DINOv2/ML.
    """
    background_color = calculate_background_color(image)
    resized_image = resize_maintaining_aspect(image, size=size)
    return apply_canvas_padding(resized_image, size=size, fill_color=background_color)


# ==============================================================================
# UTILS: CONVERSÃO LISTAS PARA DATAFRAMES
# ==============================================================================

def to_clean_dataframe(records):
    """Converte lista de dicts em pandas DataFrame limpo (ferramentas: pandas).
    
    Usos:
      - Extrair detalhes de extração para análise
      - Exportar para CSV/JSON
    
    Retorna: DataFrame vazio se lista vazia, senão DataFrame com colunas dos keys.
    """
    if not records:
        return pd.DataFrame()
    return pd.DataFrame.from_records(records)


print("✅ CONFIGURAÇÃO E UTILS PRONTAS\n")

## Módulo 2: Leitura do Excel e Gestão de Estado
Nesta secção, o notebook carrega a base de patentes, resolve hyperlinks reais e constrói o ficheiro mestre de controlo (`master_patent_tracker.csv`).

In [ ]:
# ==============================================================================
# MÓDULO 2: LEITURA DO EXCEL E GESTÃO DE ESTADO
# ==============================================================================
# RESPONSABILIDADES:
#   - Carrega modelo YOLO (YOLOv10)
#   - Lê metadados patentes de Excel (openpyxl)
#   - Extrai URLs reais (hyperlinks) vs texto visível
#   - Constrói tracker mestre (CSV) com status e metadados
#   - Devolve DataFrames limpos para módulos subsequentes
# FERRAMENTAS PRINCIPAIS:
#   - pandas: leitura Excel, DataFrames
#   - openpyxl: extração hyperlinks reais de células Excel
#   - Path: I/O ficheiros, validação paths
# ==============================================================================

print("=" * 80)
print("MÓDULO 2: LEITURA DO EXCEL E GESTÃO DE ESTADO")
print("=" * 80)


def load_patent_metadata(excel_filename, sheet_index=0, id_column="Record Number", link_column="PDF Link"):
    """Carrega metadados patentes de Excel (ferramentas: pandas, openpyxl).
    
    Etapas:
      1. find_file(): localiza ficheiro Excel em paths estruturados
      2. pd.read_excel(): carrega sheet em DataFrame
      3. Normaliza coluna ID: strip + upper (matching estável)
      4. Extrai hyperlinks reais: prioriza cell.hyperlink.target
    
    Retorna: (DataFrame de patentes, path para validação).
    """
    excel_path = find_file(excel_filename)
    df_patents = pd.read_excel(excel_path, sheet_name=sheet_index)

    if id_column in df_patents.columns:
        df_patents[id_column] = df_patents[id_column].astype(str).str.strip().str.upper()

    if link_column in df_patents.columns:
        real_urls = extract_hyperlinks_from_excel(excel_path, link_column, sheet_index=sheet_index)
        if real_urls:
            df_patents[link_column] = real_urls

    return df_patents, excel_path


def build_master_tracker(df_patents, tracker_path, id_column, link_column):
    """Constrói ou atualiza tracker mestre de patentes (ferramentas: pandas, Path, CSV).
    
    Fluxo:
      1. Itera patentes, valida URLs (http/https/ftp)
      2. Cria dicts com metadados: ID, URL, status, flags review
      3. Se tracker anterior existe: merge com histórico
      4. Preserva: duplicates, crop flags, review_status, notas do user
      5. Salva em CSV (UTF-8)
    
    Tracker columns:
      - idx, patent_id, pdf_link, link_status
      - is_duplicate, duplicate_of_patent_id
      - manual_crop_required, manual_crop_images
      - architecture_mode, architecture_stage
      - review_status (PENDING/APPROVED/REJECTED)
      - user_note, notes, last_updated
    
    Retorna: DataFrame com tracker consolidado.
    """
    records = []

    for idx, (_, row) in enumerate(df_patents.iterrows(), start=1):
        patent_id = normalize_patent_id(row.get(id_column, ""))
        pdf_link = row.get(link_column)
        if isinstance(pdf_link, str):
            pdf_link = pdf_link.strip()

        is_valid_link = isinstance(pdf_link, str) and (pdf_link.startswith("http") or pdf_link.startswith("ftp"))
        records.append(
            {
                "idx": idx,
                "patent_id": patent_id,
                "pdf_link": pdf_link if is_valid_link else "",
                "link_status": "OK" if is_valid_link else "INVALID_OR_MISSING",
                "is_duplicate": 0,
                "duplicate_of_patent_id": "",
                "manual_crop_required": 0,
                "manual_crop_images": "",
                "architecture_mode": 1,
                "architecture_stage": 0,
                "review_status": "PENDING",
                "user_note": "",
                "notes": "",
                "last_updated": "",
            }
        )

    tracker_df = pd.DataFrame.from_records(records)

    # Merge com tracker anterior para preservar histórico + metadados
    if tracker_path.exists():
        previous_master = pd.read_csv(tracker_path, dtype=str).fillna("")
        if "patent_id" in previous_master.columns:
            previous_master["patent_id"] = previous_master["patent_id"].astype(str).str.strip().str.upper()
            previous_master = previous_master.drop_duplicates(subset=["patent_id"], keep="last")

            keep_cols = [
                "patent_id",
                "is_duplicate",
                "duplicate_of_patent_id",
                "manual_crop_required",
                "manual_crop_images",
                "architecture_mode",
                "architecture_stage",
                "review_status",
                "user_note",
                "notes",
                "last_updated",
            ]

            for column_name in keep_cols:
                if column_name not in previous_master.columns:
                    previous_master[column_name] = ""

            tracker_df = tracker_df.merge(previous_master[keep_cols], on="patent_id", how="left", suffixes=("", "_old"))

            # Preserva dados antigos se não sobrescrito
            for column_name in keep_cols:
                if column_name == "patent_id":
                    continue
                old_column = f"{column_name}_old"
                if old_column in tracker_df.columns:
                    tracker_df[column_name] = tracker_df[old_column].where(
                        tracker_df[old_column].astype(str).str.strip() != "",
                        tracker_df[column_name],
                    )
                    tracker_df = tracker_df.drop(columns=[old_column])

    tracker_df = tracker_df[
        [
            "idx",
            "patent_id",
            "pdf_link",
            "link_status",
            "is_duplicate",
            "duplicate_of_patent_id",
            "manual_crop_required",
            "manual_crop_images",
            "architecture_mode",
            "architecture_stage",
            "review_status",
            "user_note",
            "notes",
            "last_updated",
        ]
    ]

    tracker_df.to_csv(tracker_path, index=False, encoding="utf-8")
    return tracker_df


# === EXECUÇÃO: MÓDULO 2 ===

print("\n1️⃣ Loading DocLayout-YOLO model...")
model = load_model_with_fallback()
print("✅ YOLO MODEL READY")

print("\n2️⃣ Loading patent data from Excel...")
excel_filename = "521_P_very_restricted_dataset_06_04_26.xlsx"
df_patents, excel_path = load_patent_metadata(
    excel_filename,
    sheet_index=EXCEL_CONFIG["sheet_index"],
    id_column=EXCEL_CONFIG["id_column"],
    link_column=EXCEL_CONFIG["link_column"],
)
print(f"   ✅ Excel: {excel_path}")
print(f"   ✅ Loaded {len(df_patents)} total records")
print(f"   Columns: {list(df_patents.columns)}")

amostra = df_patents.drop_duplicates(subset=[EXCEL_CONFIG["id_column"]]).reset_index(drop=True)
print(f"\n3️⃣ Unique patents: {len(amostra)}")

master_tracker_csv = Path.cwd() / "master_patent_tracker.csv"
master_tracker_df = build_master_tracker(
    amostra,
    master_tracker_csv,
    id_column=EXCEL_CONFIG["id_column"],
    link_column=EXCEL_CONFIG["link_column"],
)

valid_links_df = master_tracker_df.loc[master_tracker_df["link_status"] == "OK"].copy().reset_index(drop=True)
missing_links_df = master_tracker_df.loc[master_tracker_df["link_status"] != "OK"].copy().reset_index(drop=True)

print(f"   ✅ Tracker saved: {master_tracker_csv}")
print(f"   ✅ Valid links: {len(valid_links_df)}")
print(f"   ⚠️  Missing/invalid links: {len(missing_links_df)}")

print("\n4️⃣ Sample URLs and status:")
for idx, row in valid_links_df.head(3).iterrows():
    preview = row["pdf_link"][:60] + "..." if len(row["pdf_link"]) > 60 else row["pdf_link"]
    print(f"   {idx + 1}. {row['patent_id']} -> {preview}")

print("\n5️⃣ DataFrames ready for downstream modules:")
print("   - df_patents (raw metadados)")
print("   - amostra (patentes únicas)")
print("   - master_tracker_df (tracker consolidado)")
print("   - valid_links_df (apenas URLs válidas)")
print("   - missing_links_df (URLs inválidas/ausentes)")

print("\n" + "=" * 80 + "\n")

## Módulo 3: Extração, Inferência e Processamento de Imagem
Nesta secção, o notebook descarrega PDFs, executa o motor de inferência YOLO e aplica a pipeline geométrica modular antes de guardar os crops.

In [ ]:
# ==============================================================================
# MÓDULO 3: EXTRAÇÃO, INFERÊNCIA E PROCESSAMENTO DE IMAGEM
# ==============================================================================
# RESPONSABILIDADES:
#   - Download PDFs com retry
#   - Renderizar páginas PDF em imagens (PyMuPDF)
#   - Executar YOLO para detecção de objetos (DINOv2 via DocLayout-YOLO)
#   - Crop bounding boxes validadas
#   - Aplicar pipeline geométrica: cores bordas + resize + padding (PIL, cv2, numpy)
#   - Salvar crops padronizados 256x256
# FERRAMENTAS PRINCIPAIS:
#   - requests: downloads com retry
#   - pymupdf (fitz): renderizar PDF pages -> OpenCV BGR
#   - YOLOv10: detecção de objetos (classes: figure, picture, image)
#   - cv2 (OpenCV): slicing imagem, color-space
#   - PIL/Pillow: pipeline geométrica completa
#   - numpy: operações array, validações
#   - torch: detecção GPU
# ==============================================================================

print("=" * 80)
print("MÓDULO 3: EXTRAÇÃO, INFERÊNCIA E PROCESSAMENTO DE IMAGEM")
print("=" * 80)

output_dir = PATHS["output_images"]
output_dir.mkdir(exist_ok=True, parents=True)
http_session = create_http_session_with_retry()
OUTPUT_IMAGE_SIZE = 256  # DINOv2 standard input size
SKIP_ALREADY_EXTRACTED_PATENTS = True


# === RENDERIZAÇÃO PDF: FUNÇÕES AUXILIARES ===

def _batch_render_patent_pages(doc, start_page, end_page, zoom):
    """Renderiza slice de páginas PDF (ferramentas: PyMuPDF, cv2).
    
    Etapas por página:
      1. doc.load_page(page_num): carrega página PyMuPDF
      2. render_pdf_page_to_bgr(): converte para OpenCV BGR
      3. Valida se imagem não é None/vazia
      4. Acumula em batch_images com números 1-based
    
    Retorna: (lista imagens BGR, lista números página 1-based).
    """
    batch_images = []
    batch_page_numbers = []

    for page_num in range(start_page, end_page):
        try:
            page = doc.load_page(page_num)
            img_cv = render_pdf_page_to_bgr(page, zoom)
            if img_cv is None or img_cv.size == 0:
                continue
            batch_images.append(img_cv)
            batch_page_numbers.append(page_num + 1)
        except Exception:
            continue

    return batch_images, batch_page_numbers


# === PROCESSAMENTO E SALVAMENTO CROP ===

def save_processed_crop(crop_bgr, filepath, output_size=256):
    """Processa crop com pipeline geométrica e salva em disco (ferramentas: cv2, PIL).
    
    Etapas:
      1. cv2.cvtColor(): BGR (OpenCV) -> RGB (PIL standard)
      2. Image.fromarray(): cria PIL Image de array RGB
      3. process_image_pipeline(): cor bordas + resize + padding
      4. crop_final.save(): salva em PNG (lossless)
    
    Retorna: path do ficheiro gravado.
    """
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    crop_pil = Image.fromarray(crop_rgb)
    crop_final = process_image_pipeline(crop_pil, size=output_size)
    crop_final.save(filepath)
    return filepath


# === MOTOR PRINCIPAL DE EXTRAÇÃO ===

def extract_images_from_patents(
    model,
    patents_df,
    id_column,
    link_column,
    output_dir,
    session,
    skip_existing=True,
    output_size=256,
):
    """Executa pipeline completa de extração (ferramentas: requests, PyMuPDF, YOLO, PIL).
    
    Fluxo por patente:
    
    1️⃣ VALIDAÇÃO PRÉ-PROCESSAMENTO:
       - Normaliza patent_id
       - Valida URL (http/https/ftp)
       - Cria output dir
       - Skip se já extraído (flag skip_existing)
    
    2️⃣ DOWNLOAD PDF:
       - download_pdf_bytes() com retry (requests + urllib3.Retry)
       - Validações: tamanho mínimo, magic bytes %PDF
    
    3️⃣ ABERTURA E ANÁLISE PDF:
       - fitz.open(): carrega PDF em memória
       - Valida num_pages
    
    4️⃣ RENDERIZAÇÃO E DETECÇÃO (por batch):
       Para cada batch de 4 páginas:
       
       a) RENDERIZAÇÃO (PyMuPDF):
          - _batch_render_patent_pages(): renderiza páginas em OpenCV BGR
          - zoom=2.0x: 2x resolução para melhor detalhe
       
       b) INFERÊNCIA YOLO (YOLOv10):
          - run_yolo_inference_with_fallback(): batch inference
          - Se batch falha (OOM): retenta página-a-página
          - confidence_threshold: 0.35 (primary)
          - fallback_confidence_threshold: 0.18 (recovery se página vazia)
       
       c) FILTRAGEM CLASSES-ALVO:
          - extract_target_boxes(): apenas ["figure", "picture", "image"]
          - Se página sem detecções em primary: retenta com fallback conf
       
       d) POR CADA DETECÇÃO (crop):
          
          i) VALIDAÇÃO BBOX (OpenCV/NumPy):
             - sanitize_bbox(): clamp aos limites imagem, +2px pad
             - Rejeita se < min_crop_size (24px)
          
          ii) SLICING (NumPy/OpenCV):
              - crop = img_base[y1:y2, x1:x2] (numpy array slicing)
          
          iii) PROCESSAMENTO GEOMÉTRICO (PIL):
               - cv2.cvtColor(BGR -> RGB)
               - process_image_pipeline(): cor bordas + resize + padding
               - Resultado final: 256x256 com padding inteligente
          
          iv) SALVAMENTO (PIL):
               - crop_final.save(png): PNG lossless
    
    Estatísticas rastreadas:
      - pages_total, pages_rendered, pages_render_failed
      - batch_fallbacks (falhas batch retentadas)
      - detections_target (objetos encontrados)
      - invalid_crops (bboxes inválidas)
      - write_failures (erros ao salvar)
    
    Retorna: (stats dict, extraction_details_df).
    """
    stats = {
        "total_patents": len(patents_df),
        "patents_processed": 0,
        "patents_skipped_existing": 0,
        "patents_failed": 0,
        "total_images": 0,
        "details": [],
    }

    device_mode = 0 if torch.cuda.is_available() else "cpu"
    primary_conf = float(MODEL_CONFIG.get("confidence_threshold", 0.35))
    fallback_conf = float(MODEL_CONFIG.get("fallback_confidence_threshold", max(0.15, primary_conf * 0.6)))

    print(f"Inference conf: primary={primary_conf:.2f} | fallback={fallback_conf:.2f}")

    # === LOOP PRINCIPAL: PATENTE A PATENTE ===
    for patent_idx, (_, patent_row) in enumerate(patents_df.iterrows(), start=1):
        patent_id = normalize_patent_id(patent_row.get(id_column, ""))
        pdf_url = patent_row.get(link_column, "")

        print(f"\n[{patent_idx}/{stats['total_patents']}] Processing: {patent_id}")

        if isinstance(pdf_url, str):
            pdf_url = pdf_url.strip()

        # === VALIDAÇÃO PRÉ-PROCESSAMENTO ===
        if not pdf_url or not isinstance(pdf_url, str) or not (pdf_url.startswith("http") or pdf_url.startswith("ftp")):
            print(f"  ❌ Invalid URL: {pdf_url}")
            stats["patents_failed"] += 1
            stats["details"].append({"patent_id": patent_id, "status": "INVALID_URL", "images": 0})
            continue

        patent_output_dir = output_dir / patent_id
        patent_output_dir.mkdir(exist_ok=True, parents=True)

        existing_pngs = list(patent_output_dir.glob("*.png"))
        if skip_existing and existing_pngs:
            print(f"  ↪ Already extracted ({len(existing_pngs)} images). Skipping.")
            stats["patents_skipped_existing"] += 1
            stats["details"].append(
                {
                    "patent_id": patent_id,
                    "status": "SKIPPED_ALREADY_EXTRACTED",
                    "images": len(existing_pngs),
                    "pages_total": 0,
                    "pages_rendered": 0,
                    "pages_render_failed": 0,
                    "batch_fallbacks": 0,
                    "detections_target": 0,
                    "invalid_crops": 0,
                    "write_failures": 0,
                }
            )
            continue

        for old_img in existing_pngs:
            old_img.unlink(missing_ok=True)

        doc = None
        try:
            # === ETAPA 1: DOWNLOAD PDF ===
            print("  → Downloading PDF...", end=" ")
            pdf_bytes = None
            max_retries = 3

            for attempt in range(max_retries):
                pdf_bytes = download_pdf_bytes(pdf_url, session=session, timeout=PDF_CONFIG["timeout"])
                if pdf_bytes is not None:
                    break
                if attempt < max_retries - 1:
                    print(f"Retry {attempt + 1}/{max_retries}...", end=" ")

            if pdf_bytes is None:
                raise RuntimeError("Download failed after retries")

            print(f"({len(pdf_bytes) / (1024 * 1024):.1f} MB)")

            # === ETAPA 2: ABERTURA PDF ===
            doc = fitz.open(stream=pdf_bytes, filetype="pdf")
            num_pages = len(doc)
            print(f"  → PDF opened ({num_pages} pages)")

            patent_images_count = 0
            patent_debug = {
                "pages_total": num_pages,
                "pages_rendered": 0,
                "pages_render_failed": 0,
                "batch_fallbacks": 0,
                "detections_target": 0,
                "invalid_crops": 0,
                "write_failures": 0,
            }

            batch_size = int(PDF_CONFIG["batch_size"])
            zoom = float(PDF_CONFIG["zoom_extraction"])

            # === ETAPA 3: PROCESSAMENTO POR BATCH (4 páginas por batch) ===
            for batch_start in range(0, num_pages, batch_size):
                batch_end = min(batch_start + batch_size, num_pages)
                
                # === 3a) RENDERIZAÇÃO (PyMuPDF) ===
                batch_images, batch_page_numbers = _batch_render_patent_pages(doc, batch_start, batch_end, zoom)
                patent_debug["pages_rendered"] += len(batch_images)
                patent_debug["pages_render_failed"] += (batch_end - batch_start) - len(batch_images)

                if not batch_images:
                    continue

                # === 3b) INFERÊNCIA YOLO (com fallback) ===
                yolo_results, used_fallback = run_yolo_inference_with_fallback(
                    model,
                    batch_images,
                    confidence=primary_conf,
                    device=device_mode,
                )
                if used_fallback:
                    patent_debug["batch_fallbacks"] += 1

                # === 3c) PROCESSAMENTO RESULTADO YOLO (imagem a imagem no batch) ===
                for image_index, yolo_res in enumerate(yolo_results):
                    img_base = batch_images[image_index]
                    page_current = batch_page_numbers[image_index]
                    img_h, img_w = img_base.shape[:2]

                    # === 3c-i) FILTRAGEM: APENAS CLASSES-ALVO ===
                    target_boxes = extract_target_boxes(yolo_res, MODEL_CONFIG["target_classes"])
                    
                    # === 3c-ii) FALLBACK: SE PÁGINA SEM DETECÇÕES, RETENTA COM CONF MAIS BAIXA ===
                    if len(target_boxes) == 0 and fallback_conf < primary_conf:
                        single_low, _ = run_yolo_inference_with_fallback(
                            model,
                            [img_base],
                            confidence=fallback_conf,
                            device=device_mode,
                        )
                        if len(single_low) > 0:
                            target_boxes = extract_target_boxes(single_low[0], MODEL_CONFIG["target_classes"])

                    # === 3d) PROCESSAMENTO POR DETECÇÃO (crop a crop) ===
                    for obj_idx, (class_name, x1, y1, x2, y2, _) in enumerate(target_boxes):
                        patent_debug["detections_target"] += 1

                        # === 3d-i) VALIDAÇÃO BBOX ===
                        bbox = sanitize_bbox(
                            x1,
                            y1,
                            x2,
                            y2,
                            img_width=img_w,
                            img_height=img_h,
                            min_size=int(MODEL_CONFIG.get("min_crop_size", 24)),
                            pad=2,
                        )
                        if bbox is None:
                            patent_debug["invalid_crops"] += 1
                            continue

                        # === 3d-ii) SLICING (numpy array) ===
                        bx1, by1, bx2, by2 = bbox
                        crop = img_base[by1:by2, bx1:bx2]
                        if crop is None or crop.size == 0:
                            patent_debug["invalid_crops"] += 1
                            continue

                        filename = f"{patent_id}_p{page_current:03d}_o{obj_idx:02d}_{class_name}.png"
                        filepath = patent_output_dir / filename

                        # === 3d-iii) PIPELINE GEOMÉTRICA + SALVAMENTO ===
                        try:
                            save_processed_crop(crop, filepath, output_size=output_size)
                            patent_images_count += 1
                        except Exception:
                            patent_debug["write_failures"] += 1

            stats["patents_processed"] += 1
            stats["total_images"] += patent_images_count
            status = "OK" if patent_images_count > 0 else "NO_TARGET_DETECTIONS"
            stats["details"].append(
                {
                    "patent_id": patent_id,
                    "status": status,
                    "images": patent_images_count,
                    **patent_debug,
                }
            )

            print(
                f"  ✅ Extracted {patent_images_count} images "
                f"(pages ok: {patent_debug['pages_rendered']}/{patent_debug['pages_total']}, "
                f"target det: {patent_debug['detections_target']}, "
                f"invalid crops: {patent_debug['invalid_crops']})"
            )

        except Exception as e:
            stats["patents_failed"] += 1
            stats["details"].append(
                {
                    "patent_id": patent_id,
                    "status": "FAILED",
                    "images": 0,
                    "error": str(e),
                }
            )
            print(f"  ❌ Error: {str(e)[:120]}")

        finally:
            if doc is not None:
                try:
                    doc.close()
                except Exception:
                    pass

    details_df = to_clean_dataframe(stats["details"])
    return stats, details_df


# === EXECUÇÃO: MÓDULO 3 ===

extraction_stats, extraction_details_df = extract_images_from_patents(
    model=model,
    patents_df=amostra,
    id_column=EXCEL_CONFIG["id_column"],
    link_column=EXCEL_CONFIG["link_column"],
    output_dir=output_dir,
    session=http_session,
    skip_existing=SKIP_ALREADY_EXTRACTED_PATENTS,
    output_size=OUTPUT_IMAGE_SIZE,
)

print("\n" + "=" * 80)
print(
    f"EXTRACTION COMPLETE: {extraction_stats['patents_processed']} patents processed, "
    f"{extraction_stats['total_images']} images saved"
)
print("=" * 80 + "\n")

## Módulo 4: Resumo, Consolidação e Exportação
Nesta secção, os resultados da extração são consolidados em DataFrames limpos e exportados para JSON/CSV para análise posterior.

In [ ]:
# ==============================================================================
# MÓDULO 4: RESUMO, CONSOLIDAÇÃO E EXPORTAÇÃO
# ==============================================================================
# RESPONSABILIDADES:
#   - Consolida resultados de extração em estruturas limpas
#   - Gera resumo por patente (contagem crops)
#   - Merge statistics do pipeline + filesystem counts
#   - Exporta para JSON (estruturado) e CSV (tabulado)
#   - Prepara DataFrames para análise downstream
# FERRAMENTAS PRINCIPAIS:
#   - pandas: DataFrames, sorting, to_csv()
#   - json: dump estruturado
#   - pathlib: iterdir, glob para contagem ficheiros
#   - datetime: timestamp para reprodutibilidade
# ==============================================================================

print("=" * 80)
print("MÓDULO 4: RESUMO, CONSOLIDAÇÃO E EXPORTAÇÃO")
print("=" * 80)


def build_results_summary(images_dir, approved_dir):
    """Consolida resultados de extração em resumo limpo (ferramentas: pathlib, pandas).
    
    Etapas:
      1. Itera pasta output_images (subdirs = patentes)
      2. Para cada patente: conta PNG files
      3. Constrói summary dict com timestamp, totals, by_patent
      4. Cria DataFrame ordenado por (images_desc, patent_id_asc)
      5. Inclui contagem imagens aprovadas (approved_dir)
    
    Retorna: (summary dict, summary_df ordenada).
    """
    summary = {
        "timestamp": datetime.now().isoformat(),
        "totals": {
            "patents": 0,
            "images_extracted": 0,
            "images_approved": len(list(approved_dir.rglob("*.png"))),
        },
        "by_patent": [],
    }

    for patent_dir in sorted(p for p in images_dir.iterdir() if p.is_dir()):
        extracted_count = len(list(patent_dir.glob("*.png")))
        summary["totals"]["patents"] += 1
        summary["totals"]["images_extracted"] += extracted_count
        summary["by_patent"].append(
            {
                "patent_id": patent_dir.name,
                "images_extracted": extracted_count,
            }
        )

    summary_df = to_clean_dataframe(summary["by_patent"])
    if not summary_df.empty:
        # Ordena: mais images primeiro, depois alphabetically
        summary_df = summary_df.sort_values(["images_extracted", "patent_id"], ascending=[False, True]).reset_index(drop=True)

    return summary, summary_df


# === EXECUÇÃO: MÓDULO 4 ===

images_dir = PATHS["output_images"]
approved_dir = PATHS["output_approved"]
approved_dir.mkdir(exist_ok=True, parents=True)

# === BUILD SUMMARY ===
summary, summary_df = build_results_summary(images_dir, approved_dir)

# === MERGE STATISTICS ===
# extraction_stats vem do Módulo 3 (e.g., patents_processed, patents_failed)
# summary vem de filesystem (counts reais de ficheiros)
# Comparar para audit: stats vs filesystem reality
summary["details"] = extraction_details_df.to_dict(orient="records")
summary["totals"]["images_extracted_from_stats"] = int(extraction_stats.get("total_images", 0))
summary["totals"]["patents_processed_from_stats"] = int(extraction_stats.get("patents_processed", 0))
summary["totals"]["patents_failed_from_stats"] = int(extraction_stats.get("patents_failed", 0))
summary["totals"]["patents_skipped_existing_from_stats"] = int(extraction_stats.get("patents_skipped_existing", 0))

# === EXPORTAÇÃO: JSON (estruturado, completo) ===
results_file = images_dir / "extraction_results.json"
with open(results_file, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# === EXPORTAÇÃO: CSV (tabulado, por-patente) ===
summary_csv = images_dir / "extraction_results.csv"
summary_df.to_csv(summary_csv, index=False, encoding="utf-8")

# === PRINT AUDIT FINAL ===

print("\nTOTALS:")
print(f"  Patents (filesystem): {summary['totals']['patents']}")
print(f"  Images extracted (filesystem): {summary['totals']['images_extracted']}")
print(f"  Images approved: {summary['totals']['images_approved']}")
print(f"  Images extracted (stats): {summary['totals']['images_extracted_from_stats']}")
print(f"  Patents processed (stats): {summary['totals']['patents_processed_from_stats']}")
print(f"  Patents failed (stats): {summary['totals']['patents_failed_from_stats']}")
print(f"  Patents skipped (stats): {summary['totals']['patents_skipped_existing_from_stats']}")

print("\nBY PATENT:")
if summary_df.empty:
    print("  No extracted images found.")
else:
    for _, row in summary_df.head(25).iterrows():
        marker = "✅" if row["images_extracted"] > 0 else "❌"
        print(f"  {marker} {row['patent_id']:20s} -> {int(row['images_extracted']):3d} images")

print(f"\nSaved JSON: {results_file}")
print(f"Saved CSV : {summary_csv}")

print("\nClean DataFrames ready for downstream analysis:")
print("  - summary_df (resumo por patente, ordenado)")
print("  - extraction_details_df (detalhe por patente: pages_rendered, detections_target, invalid_crops, etc)")

print("\n" + "=" * 80 + "\n")